# Hangman — ensembling round 2 and round 3

No training. Rounds 2 and 3 are two independently trained networks — 8 layers
at `d_model=256` versus 10 at 384, each retrained from scratch on a different
state buffer — so they make different mistakes. Averaging their letter scores
is the cheapest remaining source of points.

`FallbackPolicy` already does the blending: it z-normalises each policy's scores
per row (so a bag-of-letters logit and a noisy-OR logit are comparable) and
mixes them by weight. `weight=1.0` is pure round 3, `0.0` is pure round 2.

| | holdout | test.txt | public LB |
|---|---|---|---|
| round 2 | 65.96% | 67.776% | 67.9306 |
| round 3 | 67.28% | 70.097% | 70.0334 |
| ensemble | ? | ? | ? |

**Stop rule:** if the best blend beats round 3 on the holdout by less than
**0.3 points**, it is inside the noise on 10,000 words — stop, do not evaluate on
test.txt, do not submit. The two-model inference cost is only worth paying for
a gain you can actually see.

Runtime: ~10 min for the sweep, ~30 min for the test set if you proceed.
Attach the competition, `hangman-src` and `hangman-weights`. GPU on.

In [ ]:
import sys, json, time, warnings
warnings.filterwarnings("ignore")

SRC = "/kaggle/input/datasets/suniljadaun/hangman-src/src"
WEIGHTS = "/kaggle/input/datasets/suniljadaun/hangman-weights"
sys.path.insert(0, SRC)

import numpy as np
import torch

from hangman.data import load_words, overlap, split_holdout, find_competition_dir
from hangman.model import HangmanNet, ModelConfig
from hangman.policies import NeuralPolicy, FallbackPolicy
from hangman.evaluate import evaluate, print_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

COMP = find_competition_dir()
train_words_all = load_words(f"{COMP}/train.txt")
test_words = load_words(f"{COMP}/test.txt")
assert overlap(train_words_all, test_words)["test_words_in_train"] == 0
MAX_LEN = max(max(map(len, train_words_all)), max(map(len, test_words)))

# Same seed as every previous run: identical 10k holdout, comparable numbers.
TRAIN_WORDS, HOLDOUT = split_holdout(train_words_all, n_holdout=10_000, seed=0)
print(len(HOLDOUT), "holdout words, MAX_LEN =", MAX_LEN)

In [ ]:
def load(name):
    ckpt = torch.load(f"{WEIGHTS}/{name}", map_location=DEVICE)
    net = HangmanNet(ModelConfig(**ckpt["cfg"]))
    net.load_state_dict(ckpt["model"])
    return net

# Each model at the fusion weight tuned for it on the holdout.
p2 = NeuralPolicy(load("hangman_r2.pt"), DEVICE, fusion=0.3)
p3 = NeuralPolicy(load("hangman_r3.pt"), DEVICE, fusion=0.6)
print("both checkpoints loaded")

## Sweep the blend weight on the full holdout

All 10,000 words, not a subsample — a 4,000-word sweep has a ~0.8-point standard
error, which is how the earlier fusion sweep ended up picking a value out of noise.

In [ ]:
results = []
for w in [1.0, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.0]:
    t = time.time()
    policy = FallbackPolicy(p3, p2, weight=w)
    m = evaluate(HOLDOUT, policy, max_len=MAX_LEN)
    results.append((m["win_rate"], -m["mean_wrong"], w))
    tag = "  <- pure round 3" if w == 1.0 else ("  <- pure round 2" if w == 0.0 else "")
    print(f"weight={w:.1f}  win {m['win_rate']:.2f}%  strikes {m['mean_wrong']:.3f}"
          f"  [{time.time()-t:.0f}s]{tag}")

best_win, neg_strikes, best_w = max(results)
solo_r3 = [r for r in results if r[2] == 1.0][0][0]
gain = best_win - solo_r3
print(f"\nbest weight {best_w} -> {best_win:.2f}%   round 3 alone {solo_r3:.2f}%"
      f"   gain {gain:+.2f}")
print("PROCEED" if gain >= 0.3 else "STOP - inside the noise, keep round 3 and do not submit")

## Only if the gain cleared 0.3

Run the next cell only when the line above says PROCEED. It plays all 250,000
test words through both networks, so it takes roughly half an hour.

In [ ]:
assert gain >= 0.3, "holdout gain is inside the noise; round 3 stays the submission"

from hangman.submit import write_submission, validate_submission

metrics = evaluate(test_words, FallbackPolicy(p3, p2, weight=best_w), max_len=MAX_LEN)
print_report(metrics)
print(f"\nround 3 alone was 70.097% / 3.433 strikes")
print(f"ensemble is    {metrics['win_rate']:.3f}% / {metrics['mean_wrong']:.3f} strikes")

write_submission(metrics["guesses"], "submission.csv")
validate_submission("submission.csv", expected_rows=len(test_words))
json.dump({"ensemble_weight": best_w, "fusion_r3": 0.6, "fusion_r2": 0.3},
          open("/kaggle/working/ensemble.json", "w"))